In [1]:
#we import things that might be useful to us, even if we do not use them or they do not appear at the end, they can be useful to test ideas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (classification_report, accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
np.random.seed(42)

In [3]:
#we choose the cancer dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
z = pd.Series(data.target, name="malignidad")

#class 0 is malignant, but that is not what we want, we need it to be the positive one for metrics like recall
y=1-z #invertimos valores
target_names=data.target_names[::-1] #invertimos clases
print("Class distribution:")
counts = y.value_counts()
percentages = y.value_counts(normalize=True) * 100
for cls, name in zip([0, 1], target_names):
    print(f" - Class {cls} ({name}): {counts[cls]} samples ({percentages[cls]:.2f}%)")
df['target']=y #lo cambiamos en nuestro dataframe

#brief initial analysis, samples, nulls, duplicates, balancing according to target
display(df.head().T)
display(f"Samples: {df.shape[0]} | Variables, without target: {df.shape[1]-1}")
display(df.dtypes)
display(df.describe().T)
display(f"Total nulls: {df.isnull().sum().sum()}")
display(f"Duplicates: {df.duplicated().sum()}")

Class distribution:
 - Class 0 (benign): 357 samples (62.74%)
 - Class 1 (malignant): 212 samples (37.26%)


,0,1,2,3,4
mean radius,17.990000,20.570000,19.690000,11.420000,20.290000
mean texture,10.380000,17.770000,21.250000,20.380000,14.340000
mean perimeter,122.800000,132.900000,130.000000,77.580000,135.100000
mean area,1001.000000,1326.000000,1203.000000,386.100000,1297.000000
mean smoothness,0.118400,0.084740,0.109600,0.142500,0.100300
mean compactness,0.277600,0.078640,0.159900,0.283900,0.132800
mean concavity,0.300100,0.086900,0.197400,0.241400,0.198000
mean concave points,0.147100,0.070170,0.127900,0.105200,0.104300
mean symmetry,0.241900,0.181200,0.206900,0.259700,0.180900
mean fractal dimension,0.078710,0.056670,0.059990,0.097440,0.058830


'Samples: 569 | Variables, without target: 30'

mean radius                float64
mean texture               float64
mean perimeter             float64
mean area                  float64
mean smoothness            float64
mean compactness           float64
mean concavity             float64
mean concave points        float64
mean symmetry              float64
mean fractal dimension     float64
radius error               float64
texture error              float64
perimeter error            float64
area error                 float64
smoothness error           float64
compactness error          float64
concavity error            float64
concave points error       float64
symmetry error             float64
fractal dimension error    float64
worst radius               float64
worst texture              float64
worst perimeter            float64
worst area                 float64
worst smoothness           float64
worst compactness          float64
worst concavity            float64
worst concave points       float64
worst symmetry      

,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


'Total nulls: 0'

'Duplicates: 0'

There are no nulls or duplicates, and all variables are numerical, so they will not need to be numerically encoded. Furthermore, the class balancing of the target variable is acceptable for this context of tumor malignancy classification. 

In [4]:
#we split the set into X features, and target
X = df.drop('target', axis=1)
y = df['target']

#70/30 with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.30, stratify=y.values, random_state=42
)

#scaling with standardscaler, we do not use it in the end but rather we assemble directly with the model to do cross-validation
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


Although we have already split into train/test, we will also assemble the scaling with the model so that when doing cross-validation it is done in each partition separately.

In [5]:
#we define a stratified cross-validation for both models
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas = ['accuracy', 'precision', 'recall', 'f1']

#logistic regression
logreg_model = LogisticRegression(l1_ratio=0, solver='liblinear', random_state=42)
#scaling pipeline and model to scale the data WITHIN each partition in an isolated way
componentes = [
    ('escalador', StandardScaler()),
    ('modelo', logreg_model)
]
pipeline_lr = Pipeline(componentes)
scores_lr = cross_validate(pipeline_lr, X_train, y_train, cv=cv_strategy, scoring=metricas, return_train_score=True)

#we show the results
display("Results for logistic regression per partition:", pd.DataFrame(scores_lr))

pipeline_lr.fit(X_train_scaled, y_train)

#random forest, we do the same as with logistic regression
rf_model = RandomForestClassifier(min_samples_leaf=5, random_state=42)
componentes = [
    ('escalador', StandardScaler()),
    ('modelo', rf_model)
]
pipeline_rf = Pipeline(componentes)
scores_rf = cross_validate(pipeline_rf, X_train, y_train, cv=cv_strategy, scoring=metricas, return_train_score=True)
#we show the results
display("Results for random forest per partition:", pd.DataFrame(scores_rf))
pipeline_rf.fit(X_train_scaled, y_train)

def mostrar_metricas(model, X_set, y_true, nombre):
    y_pred = model.predict(X_set)
    print(f"--- Metrics: {nombre} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall: {recall_score(y_true, y_pred):.3f}")
    print(f"F1-Score: {f1_score(y_true, y_pred):.3f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("-" * 30)

mostrar_metricas(pipeline_lr, X_test_scaled, y_test, "Logistic Regression")
mostrar_metricas(pipeline_rf, X_test_scaled, y_test, "Random Forest")


'Results for logistic regression per partition:'

,fit_time,score_time,test_accuracy,train_accuracy,test_precision,train_precision,test_recall,train_recall,test_f1,train_f1
0,0.017088,0.022746,0.962500,0.990566,0.935484,1.0,0.966667,0.974576,0.950820,0.987124
1,0.008199,0.017958,0.987500,0.987421,1.000000,1.0,0.966667,0.966102,0.983051,0.982759
2,0.011357,0.021247,0.987500,0.990566,1.000000,1.0,0.966667,0.974576,0.983051,0.987124
3,0.011176,0.018097,0.987342,0.987461,1.000000,1.0,0.965517,0.966387,0.982456,0.982906
4,0.016275,0.026455,0.962025,0.990596,0.964286,1.0,0.931034,0.974790,0.947368,0.987234


'Results for random forest per partition:'

,fit_time,score_time,test_accuracy,train_accuracy,test_precision,train_precision,test_recall,train_recall,test_f1,train_f1
0,0.536061,0.054822,0.900000,0.974843,0.805556,0.982456,0.966667,0.949153,0.878788,0.965517
1,0.626659,0.051701,0.975000,0.968553,1.000000,0.973684,0.933333,0.940678,0.965517,0.956897
2,0.545501,0.074094,0.975000,0.971698,0.966667,0.965812,0.966667,0.957627,0.966667,0.961702
3,0.746977,0.048552,0.898734,0.984326,0.888889,0.991379,0.827586,0.966387,0.857143,0.978723
4,0.592460,0.057933,0.949367,0.981191,0.962963,0.974790,0.896552,0.974790,0.928571,0.974790


--- Metrics: Logistic Regression ---
Accuracy: 0.971
Precision: 0.984
Recall: 0.938
F1-Score: 0.960
Confusion Matrix:
[[106   1]
 [  4  60]]
------------------------------
--- Metrics: Random Forest ---
Accuracy: 0.959
Precision: 1.000
Recall: 0.891
F1-Score: 0.942
Confusion Matrix:
[[107   0]
 [  7  57]]
------------------------------


**Which model obtains better results and with what metric do you justify it**
The best results are obtained by logistic regression in the metrics used to evaluate classification models (accuracy, recall, precision, f1). Logistic regression reaches 0.971 compared to 0.959 for the random forest, demonstrating an almost perfect balance in the model's performance. Regarding recall, it obtains 0.938 (surpassing the 0.891 of Random Forest). Therefore, given new data, the regression is superior in identifying the target class. Observing the cross-validation data, we conclude the same. The fact that logistic regression is better than random forest may be due to the fact that since we have few data, increasing the complexity of the model makes it less efficient, because generally, with greater complexity, it will need more and better data to learn to its fullest capacity.

**Which model would be easier to explain to a non-technical person?**
Logistic regression is simpler, we can explain it as a mathematical equation in which each patient variable is assigned a weight that regulates how and how much it influences the final decision. In the end, depending on whether the total score of an observation to which we apply that equation that the model has learned exceeds a threshold or not, the diagnosis is dictated. It is transparent and allows us to know exactly why the model made one decision or another.

**What metric do you consider most relevant for the problem you pose with 
your dataset and why?**
Since we are in a critical clinical environment, the relevant metric is recall (the FN in its denominator indicates the malignant ones detected as benign, which is what we most want to avoid, so if this number tends to 0, the recall tends to 1, which is why we would want to maximize it, although we also seek balance to avoid it taking the easy way out of detecting all as malignant) which is closer to 1 in the case of logistic regression (0.938) than in that of the random forest (0.891). It should be noted that although recall is more important due to sensitive and ethical issues, to also ensure the efficiency of the model it is necessary that the rest of the metrics such as precision are also high because many false positives would make the model useless for distinguishing.

**Do you observe signs of overfitting or underfitting? Justify with evidence.**
The model classifies the test data well, so we do not observe underfitting. However, overfitting does seem present due to an alarming precision of practically 1. In validation it is 1 in logistic regression and drops a 2 percent in test, while in Random Forest it remains practically stable at 1. This tells us that we would have some breathing room to demand that it further optimize the recall, which remains poor, but it might only transfer the overfitting to the negative cases. Part of the problem is naturally having few data, no matter how many variables we have to better explain the context to the model.